In [1]:
import json
import os
import grpc
import pandas as pd
from sentence_transformers import SentenceTransformer
import lancedb
from senzing import SzEngine, SzError
from senzing_grpc import SzAbstractFactoryGrpc
from senzing import SzEngineFlags

print("All imports successful")

All imports successful


## Connect to Senzing

Opens a gRPC connection to the Senzing engine using environment variables.  We need this to export the fully resolved entity data that we will turn into graph nodes and vector embeddings.

In [2]:
SENZING_HOST = os.getenv('SENZING_GRPC_HOST', 'senzing')
SENZING_PORT = os.getenv('SENZING_GRPC_PORT', '8261')

print(f"Connecting to Senzing at {SENZING_HOST}:{SENZING_PORT}")

# Create gRPC channel and engine
grpc_url = f"{SENZING_HOST}:{SENZING_PORT}"
grpc_channel = grpc.insecure_channel(grpc_url)
sz_abstract_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_abstract_factory.create_engine()

print("Connected to Senzing successfully")

Connecting to Senzing at senzing:8261
Connected to Senzing successfully


## Export All Resolved Entities

Streams the complete entity report out of Senzing, requesting the raw source record JSON alongside each resolved entity.  We need the record-level JSON because that is where names, addresses, identifiers, and risk topics are stored.

In [3]:
entities = []
export_handle = sz_engine.export_json_entity_report(
    flags=SzEngineFlags.SZ_EXPORT_INCLUDE_ALL_ENTITIES | 
          SzEngineFlags.SZ_ENTITY_INCLUDE_RECORD_JSON_DATA |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ALL_RELATIONS |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ENTITY_NAME
)

count = 0
while True:
    try:
        entity_json = sz_engine.fetch_next(export_handle)
        if not entity_json:
            break
        
        entity = json.loads(entity_json)
        entities.append(entity)
        count += 1
        
        if count % 50 == 0:
            print(f"  Exported {count} entities...", end='\r')
    except StopIteration:
        break

sz_engine.close_export_report(export_handle)
print(f"\nExported {len(entities)} entities total")

# Check how many have relationships
with_rels = sum(1 for e in entities if e.get('RELATED_ENTITIES'))
print(f"Entities with relationships: {with_rels}")

  Exported 150 entities...
Exported 196 entities total
Entities with relationships: 189


## Risk Topics Breakdown

Lists every distinct risk topic in the exported Senzing data and shows example entities for each one.  This is useful context for the workshop because it tells participants exactly what risk vocabulary the chatbot knows about before they start asking questions.

In [4]:
print("Risk Topics in Dataset:")
print("="*70)

# Extract risks and entity names directly from the Senzing export
entities_with_risks = []
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_name = entity_data.get('ENTITY_NAME', f"Entity {entity_data.get('ENTITY_ID')}")
    records = entity_data.get('RECORDS', [])

    # Determine record type from first record's FEATURES
    record_type = 'UNKNOWN'
    if records:
        for feat in records[0].get('JSON_DATA', {}).get('FEATURES', []):
            if 'RECORD_TYPE' in feat:
                record_type = feat['RECORD_TYPE']
                break

    # Collect risk topics from all records
    risks = set()
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        risk_topics = rec_json.get('risk_topics', '')
        if risk_topics:
            for t in risk_topics.split(','):
                risks.add(t.strip())
        for risk in rec_json.get('RISKS', []):
            topic = risk.get('TOPIC', '')
            if topic:
                risks.add(topic.strip())

    if risks:
        entities_with_risks.append({
            'name': entity_name,
            'type': record_type,
            'risks': risks
        })

print(f"Entities with risk data: {len(entities_with_risks)} out of {len(entities)}\n")

# Collect all unique topics
all_topics = set()
for e in entities_with_risks:
    all_topics.update(e['risks'])

for topic in sorted(all_topics):
    matching = [e for e in entities_with_risks if topic in e['risks']]
    print(f"\n{topic}: {len(matching)} entities")
    print("  Examples:")
    for e in matching[:3]:
        print(f"    - {e['name']} ({e['type']})")

Risk Topics in Dataset:
Entities with risk data: 17 out of 196


corp.disqual: 1 entities
  Examples:
    - Abassin Badshah (PERSON)

role.oligarch; role.pep; poi; sanction: 1 entities
  Examples:
    - Suleyman Abusaidovich KERIMOV (PERSON)

role.pep: 1 entities
  Examples:
    - N T Wright (PERSON)

role.rca: 1 entities
  Examples:
    - Julian Wright (PERSON)

role.rca; role.pep; sanction: 1 entities
  Examples:
    - Akhmet Magomedovich Palankoyev (PERSON)

role.rca; sanction: 4 entities
  Examples:
    - Firuza Nazimovna Kerimova (PERSON)
    - Amina Suleymanovna Kerimova (PERSON)
    - Gulnara Suleimanova KERIMOVA (PERSON)

sanction.linked: 8 entities
  Examples:
    - WANDLE HOLDINGS LIMITED (ORGANIZATION)
    - POLYUS GOLD INTERNATIONAL LIMITED (ORGANIZATION)
    - ООО "ГРАНДЕКО" (ORGANIZATION)

sanction.linked; poi: 1 entities
  Examples:
    - ООО "ГРАНДЕКО" (ORGANIZATION)


## Load the Embedding Model

Downloads and initializes `all-MiniLM-L6-v2` from Hugging Face.  This model produces 384-dimension embeddings and is a solid choice for this task since it is fast, small enough to run comfortably in the workshop environment, and handles the mix of names, addresses, and risk terminology in our entity text reasonably well.

In [5]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")
print(f"  Model: all-MiniLM-L6-v2")
print(f"  Embedding dimension: 384")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
  Model: all-MiniLM-L6-v2
  Embedding dimension: 384


## Create Entity Embeddings

Iterates over all entities and builds a text description for each one (name, type, data sources, address, identifiers, risk topics), then embeds it with the sentence transformer.  The text description is what gets stored in LanceDB and searched against at query time, so the fields included here directly determine what kinds of questions the RAG can answer well.

In [6]:
entity_data = []

for entity in entities:
    entity_data_item = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data_item.get('ENTITY_ID')
    entity_name = entity_data_item.get('ENTITY_NAME')
    records = entity_data_item.get('RECORDS', [])
    
    if not records:
        continue
    
    # Get entity info from FEATURES (v4 format)
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    features = json_data.get('FEATURES', [])
    
    name = None
    record_type = 'UNKNOWN'
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    
    # Use ENTITY_NAME from export first, then FEATURES extraction, then fallback
    name = entity_name or name or f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    
    # Get addresses from FEATURES (v4) and fallback to ADDRESSES
    addresses = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        for feat in rec_json.get('FEATURES', []):
            if feat.get('ADDR_FULL') and len(addresses) < 3:
                addresses.append(feat['ADDR_FULL'])
        for addr in rec_json.get('ADDRESSES', []):
            if addr.get('ADDR_FULL') and len(addresses) < 3:
                addresses.append(addr['ADDR_FULL'])
    
    # Get identifiers from FEATURES (v4) and fallback to IDENTIFIERS
    identifiers = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        for feat in rec_json.get('FEATURES', []):
            id_type = feat.get('NATIONAL_ID_TYPE') or feat.get('OTHER_ID_TYPE')
            id_num = feat.get('NATIONAL_ID_NUMBER') or feat.get('OTHER_ID_NUMBER')
            if id_type and id_num and len(identifiers) < 3:
                identifiers.append(f"{id_type}: {id_num}")
        for id_obj in rec_json.get('IDENTIFIERS', []):
            id_type = id_obj.get('NATIONAL_ID_TYPE') or id_obj.get('OTHER_ID_TYPE')
            id_num = id_obj.get('NATIONAL_ID_NUMBER') or id_obj.get('OTHER_ID_NUMBER')
            if id_type and id_num and len(identifiers) < 3:
                identifiers.append(f"{id_type}: {id_num}")
    
    # Get risks from top-level fields (these are outside FEATURES)
    risks = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        # Check top-level risk_topics field
        risk_topics = rec_json.get('risk_topics', '')
        if risk_topics:
            risks.extend(risk_topics.split(','))
        # Also check RISKS array
        for risk in rec_json.get('RISKS', []):
            topic = risk.get('TOPIC', '')
            if topic:
                risks.append(topic)
    
    # Get related entity info for richer text
    related = entity.get('RELATED_ENTITIES', [])
    
    # Create text description for embedding
    text_parts = [
        f"Name: {name}",
        f"Type: {record_type}",
        f"Data sources: {', '.join(data_sources)}",
        f"Records merged: {len(records)}"
    ]
    
    if addresses:
        text_parts.append(f"Address: {addresses[0]}")
    
    if identifiers:
        text_parts.append(f"Identifiers: {', '.join(identifiers[:2])}")
    
    if risks:
        text_parts.append(f"Risk topics: {', '.join(set(risks))}")
    
    if related:
        text_parts.append(f"Related to {len(related)} other entities")
    
    text = ". ".join(text_parts)
    
    # Create embedding
    embedding = embedding_model.encode(text).tolist()
    
    # Store data
    entity_data.append({
        'entity_id': entity_id,
        'name': name,
        'type': record_type,
        'text': text,
        'vector': embedding,
        'data_sources': ','.join(data_sources),
        'num_records': len(records),
        'addresses': '|'.join(addresses[:3]),
        'identifiers': '|'.join(identifiers[:3]),
        'risks': '|'.join(set(risks))
    })
    
    if len(entity_data) % 50 == 0:
        print(f"  Processed {len(entity_data)} entities...", end='\r')

print(f"\nCreated embeddings for {len(entity_data)} entities")

  Processed 150 entities...
Created embeddings for 196 entities


## Store Embeddings in LanceDB

Drops any existing `entities` table and writes all 196 entity records including their vectors into a fresh LanceDB table.  After this cell runs, the RAG notebook can connect to LanceDB and start querying without needing to touch Senzing again.

In [7]:
db = lancedb.connect('/workspace/lancedb_data')

# Drop existing table if it exists
try:
    db.drop_table('entities')
    print("Dropped existing table")
except:
    pass

# Create new table
print("Creating new table...")
table = db.create_table('entities', entity_data)

print(f"Stored {len(entity_data)} entities in LanceDB")
print("\nData preparation complete!")
print("You can now use the RAG notebook to query this data.")

Dropped existing table
Creating new table...
Stored 196 entities in LanceDB

Data preparation complete!
You can now use the RAG notebook to query this data.


## Preview LanceDB Contents

Pulls the first 10 rows from the LanceDB table and displays the key metadata columns (entity ID, name, type, data sources, record count, risks).  This is a quick gut check to confirm the data looks right before moving on.

In [8]:
print("LanceDB Contents:")
print("="*70)

sample = table.to_pandas().head(10)
display_columns = ['entity_id', 'name', 'type', 'data_sources', 'num_records', 'risks']
print(sample[display_columns].to_string())

LanceDB Contents:
   entity_id                               name          type                   data_sources  num_records                                 risks
0     100001                    Abassin Badshah        PERSON  OPEN-OWNERSHIP,OPEN-SANCTIONS            3                          corp.disqual
1     100002                        LMAR GB LTD  ORGANIZATION                 OPEN-SANCTIONS            1                                      
2     100003            WANDLE HOLDINGS LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1                       sanction.linked
3     100004  POLYUS GOLD INTERNATIONAL LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1                       sanction.linked
4     100005          Firuza Nazimovna Kerimova        PERSON                 OPEN-SANCTIONS            1                    role.rca; sanction
5     100006                     ООО "ГРАНДЕКО"  ORGANIZATION                 OPEN-SANCTIONS            2  sanction.li

## LanceDB Statistics

Prints summary counts of what is in the vector store: total entities, breakdown by type (PERSON vs ORGANIZATION), breakdown by data source, and a count of entities that carry at least one risk flag.  Only 17 out of 196 entities have risk data, which reflects the relative size of the OPEN-SANCTIONS dataset.

In [9]:
all_entities = table.to_pandas()

print("\nLanceDB Statistics:")
print("="*70)
print(f"Total entities: {len(all_entities)}")
print()

print("By Type:")
print(all_entities['type'].value_counts())
print()

print("By Data Source:")
print(all_entities['data_sources'].value_counts())
print()

print("Entities with risks/sanctions:")
has_risks = all_entities[all_entities['risks'].notna() & (all_entities['risks'] != '')]
print(f"  Count: {len(has_risks)}")


LanceDB Statistics:
Total entities: 196

By Type:
type
ORGANIZATION    127
PERSON           69
Name: count, dtype: int64

By Data Source:
data_sources
OPEN-OWNERSHIP                   176
OPEN-SANCTIONS                    17
OPEN-OWNERSHIP,OPEN-SANCTIONS      3
Name: count, dtype: int64

Entities with risks/sanctions:
  Count: 17


## Inspect Sample Entity Records

Prints the full stored record for the first 5 entities including the first and last few values of each embedding vector.  This gives participants a concrete look at exactly what shape of data is going into LanceDB and being searched at query time.

In [10]:
sample = table.to_pandas().head(5)

# Convert to dict and display as formatted JSON
for idx, row in sample.iterrows():
    print(f"\nEntity {idx + 1}:")
    print("="*70)
    
    # Convert row to dict
    row_dict = row.to_dict()
    
    # Show vector info separately
    vector = row_dict.pop('vector')
    
    # Print everything except vector
    print(json.dumps(row_dict, indent=2))
    
    # Print vector summary
    print(f"\nvector: [{vector[0]}, {vector[1]}, {vector[2]}, ..., {vector[-1]}]")
    print(f"  (384-dimension vector, first few values shown)")
    print()


Entity 1:
{
  "entity_id": 100001,
  "name": "Abassin Badshah",
  "type": "PERSON",
  "text": "Name: Abassin Badshah. Type: PERSON. Data sources: OPEN-OWNERSHIP, OPEN-SANCTIONS. Records merged: 3. Address: 31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL. Identifiers: OPEN-SANCTIONS: NK-25vyVFzt8vdJGgAXMRTwTJ. Risk topics: corp.disqual. Related to 5 other entities",
  "data_sources": "OPEN-OWNERSHIP,OPEN-SANCTIONS",
  "num_records": 3,
  "addresses": "31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL|31, Quernmore Close, Bromley, BR1 4EL|3, Market Parade, 41 East Street, Bromley, BR1 1QN",
  "identifiers": "OPEN-SANCTIONS: NK-25vyVFzt8vdJGgAXMRTwTJ",
  "risks": "corp.disqual"
}

vector: [0.01783916726708412, -0.02757100947201252, 0.006671417038887739, ..., -0.029885638505220413]
  (384-dimension vector, first few values shown)


Entity 2:
{
  "entity_id": 100002,
  "name": "LMAR GB LTD",
  "type": "ORGANIZATION",
  "text": "Name: LMAR GB LTD. Type: ORGANIZATION. Dat

In [11]:
print("Closing connections...")

try:
    grpc_channel.close()
    print("Done")
except Exception as e:
    print(f"Note: {e}")

Closing connections...
Done
